# Company Quarter Prompt Search

In [1]:
import sys
from pathlib import Path

project_root = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "src" / "data_collection" / "consts.py").is_file()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Could not locate project root containing 'src' directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM

from src.data_collection.consts import DB_PARAMS
from src.data_collection.model_driver.model_driver_class import ModelDriver
from src.data_analysis.data_fetcher.data_fetcher_class import DataFetcher

/home/maxim-shibanov/anaconda3/envs/vllm_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
fq_verbolizer = {
    1: ["first", "january", "february", "march", "spring", "earlyyear"],
    2: ["second", "april", "may", "june", "midyear"],
    3: ["third", "july", "august", "september", "lateyear"],
    4: ["fourth", "october", "november", "december", "yearend", "annual", "yearly"],
}

quarter_prompts = [
    "Read the report excerpt and identify the calendar quarter of the SEC filing date. Selected quarter:",
    "Classify the calendar quarter of the filing date for this SEC report. Classification:",
    "You are tagging SEC filings by submission quarter. From this report excerpt, the filing belongs to the",
    "Imagine you are reviewing SEC metadata and need to assign the filing to a calendar quarter. The correct quarter is",
    "Treat this as a metadata classification task for an SEC document. Your goal is not to identify the reporting period quarter, but the actual calendar quarter in which the filing was submitted to the SEC. Based on the excerpt, the filing quarter should be labeled:",
    "Read the report excerpt and identify this company's quarter within its reporting year. Selected company quarter:", # The best 
    "Classify the company quarter rank for this SEC report within the company year. Classification:",
    "You are tagging company reports by quarter rank within the company year. From this report excerpt, the company quarter is",
    "Imagine you are reviewing company reporting metadata and need to assign the report to the correct company quarter within the year. The correct company quarter is",
    "Treat this as a metadata classification task for an SEC document. Your goal is to identify the company's quarter rank within its reporting year. Based on the excerpt, the company quarter should be labeled:",
    "Using the excerpt, infer where this report falls in the company's own yearly reporting sequence. Choose among first, second, third, and fourth. The company quarter rank is:",
    "Ignore the SEC submission month and focus on the company's reporting cycle. Which company quarter does this report represent within the year? Answer:",
    "Determine the quarter number of this report within the company's annual reporting pattern, rather than the calendar filing quarter. The correct company quarter is:",
    "Based on the reporting language and report structure, assign this filing to the company's first, second, third, or fourth reporting quarter of the year. Company quarter:",
    "You are labeling reports by quarter order within each company's reporting year. From this excerpt, the most likely company quarter rank is:",
]

In [4]:
model_name = "mistralai/Mistral-7B-v0.1"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    max_memory={0: "12GiB", "cpu": "64GiB"},
)

model_driver = ModelDriver(model_name=model_name, model=model)

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.13it/s]
2026-05-26 12:15:18,929 [WARNING] Some parameters are on the meta device because they were offloaded to the cpu.


In [5]:
def cyclic_delta(pred_quarter: int, true_quarter: int) -> int:
    return ((pred_quarter - true_quarter + 2) % 4) - 2

def shift_quarter_label(quarter: int, step: int = 1) -> int:
    return ((quarter - 1 + step) % 4) + 1


fetcher = DataFetcher(DB_PARAMS)
reports_df = fetcher._fetch_reports(regressors=["raw_text"])
reports_df = reports_df[reports_df["report_type"].isin(["10-K", "10-Q"])].copy()
reports_df["filed_date"] = pd.to_datetime(reports_df["filed_date"])
reports_df = reports_df.sort_values("filed_date").copy()
reports_df["year"] = reports_df["filed_date"].dt.year
reports_df["filing_rank"] = reports_df.groupby(["cik", "year"]).cumcount()
reports_df = reports_df[
    reports_df["filing_rank"] >= (
        reports_df.groupby(["cik", "year"])["filing_rank"].transform("max") - 3
    )
].copy()
reports_df["true_label"] = reports_df.groupby(["cik", "year"]).cumcount() + 1

sample_q1 = reports_df[reports_df["true_label"] == 1].sample(n=10, random_state=42)
sample_q2 = reports_df[reports_df["true_label"] == 2].sample(n=10, random_state=42)
sample_q3 = reports_df[reports_df["true_label"] == 3].sample(n=10, random_state=42)
sample_q4 = reports_df[reports_df["true_label"] == 4].sample(n=10, random_state=42)

sample_df = pd.concat([sample_q1, sample_q2, sample_q3, sample_q4], ignore_index=True)
sample_df = sample_df.sample(frac=1.0, random_state=42).reset_index(drop=True)

sample_df[["id", "report_type", "filed_date", "true_label"]].head()

/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:111: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:92: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:130: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2

Available regressors:
 - avg_default_verbolizer
 - avg_shrink_verbolizer
 - doc_len
 - eps_surprise
 - f_size
 - full_list_default_verbolizer
 - full_list_shrink_verbolizer
 - hv_orig_score
 - lm_orig_score
 - max_abs_default
 - max_abs_shrink
 - max_default_verbolizer
 - max_shrink_verbolizer
 - md_hv1
 - md_hv2
 - md_hv3
 - md_lm1
 - md_lm2
 - md_lm3
 - min_default_verbolizer
 - min_shrink_verbolizer
 - stretch_default
 - stretch_shrink
Available sectors:
 - Technology (92)
 - Industrials (86)
 - Financial Services (85)
 - Healthcare (66)
 - Consumer Cyclical (58)
 - Consumer Defensive (40)
 - Real Estate (32)
 - Utilities (32)
 - Energy (30)
 - Basic Materials (23)
 - Communication Services (22)


,id,report_type,filed_date,true_label
0,88361,10-Q,2022-04-29,2
1,26105,10-Q,2019-04-22,2
2,22304,10-Q,2019-05-03,2
3,30847,10-Q,2019-07-23,3
4,103015,10-K,2023-02-24,1


In [10]:
def evaluate_prompt(prompt: str, data: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    rows = []
    for row in tqdm(data.itertuples(index=False), total=len(data), desc=prompt[:48]):
        result = model_driver.predict_metadata(
            verbolizer=fq_verbolizer,
            prompt=prompt,
            text=row.raw_text,
        )
        probabilities = result["probabilities"]
        predicted_label = max(probabilities, key=probabilities.get)
        rows.append(
            {
                "id": row.id,
                "report_type": row.report_type,
                "filed_date": row.filed_date,
                "true_label": row.true_label,
                "predicted_label": predicted_label,
                "confidence": result["confidence"],
                "true_label_probability": probabilities[row.true_label],
            }
        )

    predictions_df = pd.DataFrame(rows)
    predictions_df["cyclic_delta"] = predictions_df.apply(
        lambda row: cyclic_delta(row["predicted_label"], row["true_label"]),
        axis=1,
    )
    predictions_df["shifted_predicted_label"] = predictions_df["predicted_label"].map(
        lambda label: shift_quarter_label(label, step=1)
    )
    y_true = predictions_df["true_label"]
    y_pred = predictions_df["predicted_label"]
    y_pred_shifted = predictions_df["shifted_predicted_label"]

    metrics = {
        "prompt": prompt,
        "avg_confidence": predictions_df["confidence"].mean(),
        "avg_true_label_probability": predictions_df["true_label_probability"].mean(),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "avg_cyclic_delta": predictions_df["cyclic_delta"].mean(),
        "shifted_accuracy": accuracy_score(y_true, y_pred_shifted),
        "shifted_f1_macro": f1_score(y_true, y_pred_shifted, average="macro", zero_division=0),
    }
    return metrics, predictions_df


all_metrics = []
all_predictions = {}

for prompt in quarter_prompts:
    metrics, predictions_df = evaluate_prompt(prompt, sample_df)
    all_metrics.append(metrics)
    all_predictions[prompt] = predictions_df

    print("\nPROMPT:", prompt)
    print(predictions_df["predicted_label"].value_counts())
    print(predictions_df["cyclic_delta"].value_counts().sort_index())
    print(pd.crosstab(predictions_df["true_label"], predictions_df["predicted_label"]))

results_df = pd.DataFrame(all_metrics)
results_df = results_df.sort_values(["shifted_f1_macro", "avg_confidence"], ascending=[False, False]).reset_index(drop=True)
display(results_df)

Read the report excerpt and identify the calenda: 100%|██████████| 40/40 [01:36<00:00,  2.42s/it]



PROMPT: Read the report excerpt and identify the calendar quarter of the SEC filing date. Selected quarter:
predicted_label
4    14
3    13
1     9
2     4
Name: count, dtype: int64
cyclic_delta
-2     3
-1    31
 0     5
 1     1
Name: count, dtype: int64
predicted_label  1  2  3   4
true_label                  
1                0  0  0  10
2                8  0  0   2
3                1  4  4   1
4                0  0  9   1


Classify the calendar quarter of the filing date: 100%|██████████| 40/40 [01:34<00:00,  2.35s/it]



PROMPT: Classify the calendar quarter of the filing date for this SEC report. Classification:
predicted_label
4    31
1     6
3     3
Name: count, dtype: int64
cyclic_delta
-2    11
-1    12
 0     8
 1     9
Name: count, dtype: int64
predicted_label  1  3  4
true_label              
1                1  1  8
2                2  0  8
3                2  0  8
4                1  2  7


You are tagging SEC filings by submission quarte: 100%|██████████| 40/40 [01:39<00:00,  2.49s/it]


...

Based on the reporting language and report struc: 100%|██████████| 40/40 [01:40<00:00,  2.52s/it]



PROMPT: Based on the reporting language and report structure, assign this filing to the company's first, second, third, or fourth reporting quarter of the year. Company quarter:
predicted_label
4    24
1    16
Name: count, dtype: int64
cyclic_delta
-2     8
-1    13
 0    10
 1     9
Name: count, dtype: int64
predicted_label  1  4
true_label           
1                3  7
2                6  4
3                4  6
4                3  7


You are labeling reports by quarter order within: 100%|██████████| 40/40 [01:39<00:00,  2.49s/it]


PROMPT: You are labeling reports by quarter order within each company's reporting year. From this excerpt, the most likely company quarter rank is:
predicted_label
1    30
3     6
4     3
2     1
Name: count, dtype: int64
cyclic_delta
-2    10
-1    15
 0    10
 1     5
Name: count, dtype: int64
predicted_label  1  2  3  4
true_label                 
1                7  0  1  2
2                9  1  0  0
3                9  0  1  0
4                5  0  4  1


,prompt,avg_confidence,avg_true_label_probability,accuracy,precision_macro,recall_macro,f1_macro,avg_cyclic_delta,shifted_accuracy,shifted_f1_macro
0,You are labeling reports by quarter order with...,0.047536,0.244551,0.250,0.433333,0.250,0.202666,-0.750,0.375,0.314423
1,You are tagging company reports by quarter ran...,0.074653,0.216991,0.200,0.194498,0.200,0.195627,-0.825,0.775,0.774307
2,Imagine you are reviewing SEC metadata and nee...,0.125270,0.218845,0.200,0.196825,0.200,0.194675,-0.800,0.725,0.724892
3,Determine the quarter number of this report wi...,0.097246,0.257047,0.225,0.150000,0.225,0.175000,-0.650,0.450,0.383333
4,Imagine you are reviewing company reporting me...,0.071755,0.207243,0.175,0.175267,0.175,0.173888,-0.800,0.775,0.774307
5,Based on the reporting language and report str...,0.105318,0.239647,0.250,0.119792,0.250,0.160633,-0.500,0.325,0.218326
6,Read the report excerpt and identify this comp...,0.078333,0.212463,0.150,0.149432,0.150,0.149206,-0.875,0.825,0.823016
7,Treat this as a metadata classification task f...,0.136723,0.219166,0.150,0.145563,0.150,0.144345,-0.850,0.775,0.765586
8,Ignore the SEC submission month and focus on t...,0.114110,0.224349,0.150,0.141270,0.150,0.142043,-0.750,0.800,0.797523
9,Treat this as a metadata classification task f...,0.061226,0.238143,0.200,0.104688,0.200,0.116667,-0.625,0.375,0.306169


In [11]:
results_df = results_df.sort_values(["shifted_f1_macro", "avg_confidence"], ascending=[False, False]).reset_index(drop=True)
display(results_df)

,prompt,avg_confidence,avg_true_label_probability,accuracy,precision_macro,recall_macro,f1_macro,avg_cyclic_delta,shifted_accuracy,shifted_f1_macro
0,Read the report excerpt and identify this comp...,0.078333,0.212463,0.150,0.149432,0.150,0.149206,-0.875,0.825,0.823016
1,Ignore the SEC submission month and focus on t...,0.114110,0.224349,0.150,0.141270,0.150,0.142043,-0.750,0.800,0.797523
2,You are tagging company reports by quarter ran...,0.074653,0.216991,0.200,0.194498,0.200,0.195627,-0.825,0.775,0.774307
3,Imagine you are reviewing company reporting me...,0.071755,0.207243,0.175,0.175267,0.175,0.173888,-0.800,0.775,0.774307
4,Treat this as a metadata classification task f...,0.136723,0.219166,0.150,0.145563,0.150,0.144345,-0.850,0.775,0.765586
5,Read the report excerpt and identify the calen...,0.125490,0.213277,0.125,0.094780,0.125,0.107790,-0.900,0.775,0.757369
6,Imagine you are reviewing SEC metadata and nee...,0.125270,0.218845,0.200,0.196825,0.200,0.194675,-0.800,0.725,0.724892
7,You are tagging SEC filings by submission quar...,0.085657,0.218724,0.175,0.058333,0.175,0.087500,-0.625,0.500,0.498626
8,Determine the quarter number of this report wi...,0.097246,0.257047,0.225,0.150000,0.225,0.175000,-0.650,0.450,0.383333
9,You are labeling reports by quarter order with...,0.047536,0.244551,0.250,0.433333,0.250,0.202666,-0.750,0.375,0.314423


In [14]:
top_3_prompts = results_df.head(3).copy()
display(top_3_prompts[["prompt", "avg_confidence", "precision_macro", "recall_macro", "f1_macro", "accuracy", "avg_cyclic_delta", "shifted_accuracy", "shifted_f1_macro"]])

for idx, row in top_3_prompts.iterrows():
    print(f"\nTop {idx + 1} prompt")
    print(row["prompt"])
    print(
        {
            "avg_confidence": row["avg_confidence"],
            "precision_macro": row["precision_macro"],
            "recall_macro": row["recall_macro"],
            "f1_macro": row["f1_macro"],
            "accuracy": row["accuracy"],
            "avg_cyclic_delta": row["avg_cyclic_delta"],
            "shifted_accuracy": row["shifted_accuracy"],
            "shifted_f1_macro": row["shifted_f1_macro"],
        }
    )

,prompt,avg_confidence,precision_macro,recall_macro,f1_macro,accuracy,avg_cyclic_delta,shifted_accuracy,shifted_f1_macro
0,Read the report excerpt and identify this comp...,0.078333,0.149432,0.15,0.149206,0.15,-0.875,0.825,0.823016
1,Ignore the SEC submission month and focus on t...,0.114110,0.141270,0.15,0.142043,0.15,-0.750,0.800,0.797523
2,You are tagging company reports by quarter ran...,0.074653,0.194498,0.20,0.195627,0.20,-0.825,0.775,0.774307



Top 1 prompt
Read the report excerpt and identify this company's quarter within its reporting year. Selected company quarter:
{'avg_confidence': 0.0783326683100313, 'precision_macro': 0.14943181818181817, 'recall_macro': 0.15000000000000002, 'f1_macro': 0.1492063492063492, 'accuracy': 0.15, 'avg_cyclic_delta': -0.875, 'shifted_accuracy': 0.825, 'shifted_f1_macro': 0.823015873015873}

Top 2 prompt
Ignore the SEC submission month and focus on the company's reporting cycle. Which company quarter does this report represent within the year? Answer:
{'avg_confidence': 0.11411038362421096, 'precision_macro': 0.14126984126984127, 'recall_macro': 0.15000000000000002, 'f1_macro': 0.14204334365325078, 'accuracy': 0.15, 'avg_cyclic_delta': -0.75, 'shifted_accuracy': 0.8, 'shifted_f1_macro': 0.7975232198142416}

Top 3 prompt
You are tagging company reports by quarter rank within the company year. From this report excerpt, the company quarter is
{'avg_confidence': 0.07465294197318144, 'precision_ma

In [8]:
# Optional: inspect one prediction with top-token printing.
example_prompt = top_3_prompts.iloc[0]["prompt"]
example_report = sample_df.iloc[0]["raw_text"]
model_driver.predict_metadata(
    verbolizer=fq_verbolizer,
    prompt=example_prompt,
    text=example_report,
    print_top_tokens=True,
    top_k_tokens=20,
)


Top 20 tokens by probability:
Quarter      | 0.103027
March        | 0.048584
Third        | 0.029419
Fourth       | 0.016846
End          | 0.014343
Select       | 0.011169
December     | 0.009888
SEC          | 0.009583
April        | 0.007446
September    | 0.006775
January      | 0.006775
Ex           | 0.006165
June         | 0.005798
Date         | 0.005615
Current      | 0.005463
quarter      | 0.005280
Dec          | 0.004669
SE           | 0.004242
Click        | 0.004120
end          | 0.003860


{'confidence': 0.1454448401927948,
 'probabilities': {1: 0.39796189552363276,
  3: 0.2821336064302354,
  4: 0.22558320470444657,
  2: 0.09432129334168525}}